In [1]:
import networkx as nx

# 그래프 생성
G = nx.Graph()

# 동일한 이름의 노드를 구분하여 추가
G.add_node("Node1_1", label="Node1")
G.add_node("Node1_2", label="Node1")
G.add_node("Node2", label="Node2")

# 엣지 추가
G.add_edge("Node1_1", "Node2")
G.add_edge("Node1_2", "Node2")

# 노드와 엣지 확인
print("Nodes:", G.nodes(data=True))
print("Edges:", G.edges())

Nodes: [('Node1_1', {'label': 'Node1'}), ('Node1_2', {'label': 'Node1'}), ('Node2', {'label': 'Node2'})]
Edges: [('Node1_1', 'Node2'), ('Node1_2', 'Node2')]


In [1]:
class Subtask:
    def __init__(self, name, duration, constraints):
        self.name = name
        self.duration = duration
        self.constraints = constraints


class TaskNode:
    def __init__(self, state, completed_subtasks):
        self.state = state  # state of the world, e.g., time
        self.completed_subtasks = completed_subtasks  # list of completed subtasks
        self.children = []

    def is_constraint_satisfied(self, constraint, completed_subtasks):
        if constraint["Type"] == "After":
            required_subtask = constraint["Subtask"]
            if required_subtask is None:
                return True
            return required_subtask in completed_subtasks
        elif constraint["Type"] == "Before":
            required_subtask = constraint["Subtask"]
            if required_subtask is None:
                return True
            return required_subtask not in completed_subtasks
        return True

    def expand(self, all_subtasks):
        for subtask in all_subtasks:
            if subtask.name not in self.completed_subtasks:
                if all(
                    self.is_constraint_satisfied(c, self.completed_subtasks)
                    for c in subtask.constraints
                ):
                    new_state = self.state + subtask.duration
                    new_completed_subtasks = self.completed_subtasks + [subtask.name]
                    child_node = TaskNode(new_state, new_completed_subtasks)
                    self.children.append(child_node)
                    child_node.expand(all_subtasks)


def create_subtasks():
    cook_steak_subtasks = [
        Subtask(
            "Turn on Stove",
            5,
            [{"Type": "After", "Subtask": None, "Interval": 0, "Urgency": False}],
        ),
        Subtask(
            "Placing Steak",
            5,
            [
                {
                    "Type": "After",
                    "Subtask": "Turn on Stove",
                    "Interval": 5,
                    "Urgency": True,
                }
            ],
        ),
        Subtask(
            "Monitoring Steak",
            20,
            [
                {
                    "Type": "After",
                    "Subtask": "Placing Steak",
                    "Interval": 0,
                    "Urgency": True,
                }
            ],
        ),
        Subtask(
            "Flipping Steak",
            5,
            [
                {
                    "Type": "After",
                    "Subtask": "Monitoring Steak",
                    "Interval": 0,
                    "Urgency": True,
                }
            ],
        ),
        Subtask("Seasoning Steak", 5, []),
    ]

    clean_kitchen_subtasks = [
        Subtask(
            "Washing Dishes",
            10,
            [{"Type": "After", "Subtask": None, "Interval": 0, "Urgency": False}],
        ),
        Subtask(
            "Setting Table",
            10,
            [
                {
                    "Type": "After",
                    "Subtask": "Toast End",
                    "Interval": 0,
                    "Urgency": False,
                },
                {
                    "Type": "After",
                    "Subtask": "Flipping Steak",
                    "Interval": 0,
                    "Urgency": False,
                },
                {
                    "Type": "Before",
                    "Subtask": "Seasoning Steak",
                    "Interval": 0,
                    "Urgency": False,
                },
            ],
        ),
    ]

    toast_subtasks = [
        Subtask(
            "Toast Start",
            1,
            [{"Type": "After", "Subtask": None, "Interval": 0, "Urgency": False}],
        ),
        Subtask(
            "Toast End",
            1,
            [
                {
                    "Type": "After",
                    "Subtask": "Toast Start",
                    "Interval": 5,
                    "Urgency": True,
                }
            ],
        ),
    ]

    return cook_steak_subtasks + clean_kitchen_subtasks + toast_subtasks


def main():
    subtasks = create_subtasks()
    root = TaskNode(0, [])
    root.expand(subtasks)

    def print_tree(node, level=0):
        print(
            " " * (level * 2)
            + f"State: {node.state}, Completed: {node.completed_subtasks}"
        )
        for child in node.children:
            print_tree(child, level + 1)

    print_tree(root)


if __name__ == "__main__":
    main()

State: 0, Completed: []
  State: 5, Completed: ['Turn on Stove']
    State: 10, Completed: ['Turn on Stove', 'Placing Steak']
      State: 30, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak']
        State: 35, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak', 'Flipping Steak']
          State: 40, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak', 'Flipping Steak', 'Seasoning Steak']
            State: 50, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak', 'Flipping Steak', 'Seasoning Steak', 'Washing Dishes']
              State: 51, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak', 'Flipping Steak', 'Seasoning Steak', 'Washing Dishes', 'Toast Start']
                State: 52, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring Steak', 'Flipping Steak', 'Seasoning Steak', 'Washing Dishes', 'Toast Start', 'Toast End']
            State: 41, Completed: ['Turn on Stove', 'Placing Steak', 'Monitoring St

In [2]:
from anytree import Node, RenderTree
from anytree.exporter import UniqueDotExporter

# 트리 생성
root = Node("root")
a = Node("a", parent=root)
b = Node("b", parent=root)
c = Node("c", parent=a)
d = Node("d", parent=a)
e = Node("e", parent=b)
f = Node("f", parent=b)

# 강조하고자 하는 경로
highlight_path = [root, a, d]  # Set 사용으로 성능 향상


# nodeattrfunc 정의
def nodeattrfunc(node):
    if node in highlight_path:
        return 'color="red", style=filled, fillcolor=lightgrey'
    return 'color="black"'


# UniqueDotExporter 사용
UniqueDotExporter(root, nodeattrfunc=nodeattrfunc).to_picture("tree.png")

# 트리 출력
for pre, fill, node in RenderTree(root):
    print("%s%s" % (pre, node.name))

root
├── a
│   ├── c
│   └── d
└── b
    ├── e
    └── f


In [3]:
from anytree import Node
from anytree.exporter import UniqueDotExporter
from typing import List

def visualize_tree(tree: Node, plans: List[List[Node]]):
    def nodeattrfunc(node):
        for plan in plans:
            if node in plan:
                return 'color="red", style=filled, fillcolor=lightgrey'
        return 'color="black"'
    
    # edge의 기본 스타일을 설정
    def edgeattrfunc(node, child):
        return 'color="black"'
    
    # 노드 이름을 문자열로 사용하도록 명시
    def nodenamefunc(node):
        return node.name

    UniqueDotExporter(
        tree, 
        nodeattrfunc=nodeattrfunc, 
        nodenamefunc=nodenamefunc,
        edgeattrfunc=edgeattrfunc
    ).to_picture("task_tree.png")

# 예제 트리 생성
root = Node("root")
a = Node("a", parent=root)
b = Node("b", parent=root)
c = Node("c", parent=a)
d = Node("d", parent=a)
e = Node("e", parent=b)
f = Node("f", parent=b)

# 강조하고자 하는 경로들
highlight_paths = [[root, a, d], [root, b, e]]

# 트리 시각화 호출
visualize_tree(root, highlight_paths)
